# mem7 Demo — Ollama + Upstash Vector

This notebook demonstrates the full mem7 memory pipeline:
- **LLM**: Ollama (qwen2.5:7b) — local, OpenAI-compatible
- **Embedding**: Ollama (mxbai-embed-large) — 1024 dims
- **Vector Store**: Upstash Vector (REST API)
- **History**: SQLite (in-memory)

## 1. Prerequisites

Make sure Ollama is running and models are pulled:
```bash
ollama serve                    # start the server (if not already running)
ollama pull qwen2.5:7b          # LLM
ollama pull mxbai-embed-large   # Embedding (1024 dims)
```

In [ ]:
from mem7 import Memory
from mem7.config import MemoryConfig, LlmConfig, EmbeddingConfig, VectorConfig, HistoryConfig
import json

def pp(obj):
    """Pretty-print a dict/list."""
    print(json.dumps(obj, indent=2, ensure_ascii=False))

## 2. Configure mem7

Using **Ollama** for LLM + Embedding, and **Upstash Vector** as the vector store.

In [ ]:
import os
from mem7.config import ContextConfig

config = MemoryConfig(
    llm=LlmConfig(
        provider="ollama",
        base_url="http://localhost:11434/v1",
        api_key="ollama",
        model="qwen2.5:7b",
        temperature=0.0,
        max_tokens=2000,
    ),
    embedding=EmbeddingConfig(
        provider="ollama",
        base_url="http://localhost:11434/v1",
        api_key="ollama",
        model="mxbai-embed-large",
        dims=1024,
    ),
    vector=VectorConfig(
        provider="upstash",
        collection_name="mem7-test",
        dims=1024,
        upstash_url=os.environ["UPSTASH_VECTOR_REST_URL"],
        upstash_token=os.environ["UPSTASH_VECTOR_REST_TOKEN"],
    ),
    history=HistoryConfig(db_path=":memory:"),
    context=ContextConfig(enabled=True),
)

m = Memory(config=config)
print("mem7 initialized ✓ (context-aware scoring enabled)")

## 3. Reset (clean slate)

Clear any previous data in the Upstash namespace so we start fresh.

In [ ]:
m.reset()
print("reset done ✓")

## 4. Add memories from conversations

The `add()` method accepts conversation messages. mem7 will:
1. Use the LLM to extract factual statements
2. Embed each fact
3. Search for existing similar memories
4. Use the LLM to decide ADD / UPDATE / DELETE / NONE
5. Persist to Upstash Vector + SQLite history

In [ ]:
result = m.add(
    messages=[
        {"role": "user", "content": "I'm working on improving my tennis skills. I play twice a week."},
        {"role": "assistant", "content": "That's great! Consistent practice is key. What areas are you focusing on?"},
        {"role": "user", "content": "I'm focusing on my backhand and serve. I also recently started taking lessons from a coach named Sarah."},
    ],
    user_id="alice",
)
print("=== Add result ===")
pp(result)

In [ ]:
result2 = m.add(
    messages=[
        {"role": "user", "content": "I recently moved to San Francisco and I love hiking in Marin."},
    ],
    user_id="alice",
)
print("=== Add result 2 ===")
pp(result2)

## 5. Search memories

Semantic search — finds memories most relevant to the query, filtered by user_id.

In [ ]:
results = m.search("What sports does Alice play?", user_id="alice", limit=3)
print("=== Search results ===")
pp(results)

## 6. Get all memories

In [ ]:
all_memories = m.get_all(user_id="alice")
print(f"Total memories for alice: {len(all_memories)}")
pp(all_memories)

## 7. Update a memory (deduplication)

Adding overlapping facts — the LLM should decide to UPDATE existing memories rather than ADD duplicates.

In [ ]:
result3 = m.add(
    messages=[
        {"role": "user", "content": "I now play tennis three times a week and my coach Sarah says my backhand is improving a lot."},
    ],
    user_id="alice",
)
print("=== Add with dedup ===")
pp(result3)

In [ ]:
print("=== Memories after dedup ===")
all_after = m.get_all(user_id="alice")
print(f"Total memories: {len(all_after)}")
pp(all_after)

## 8. View history

Every ADD / UPDATE / DELETE action is recorded in the SQLite audit trail.

In [ ]:
if all_after:
    first_id = all_after[0]["id"]
    history = m.history(first_id)
    print(f"=== History for memory {first_id} ===")
    pp(history)
else:
    print("No memories to show history for")

## 9. Manual update & delete

In [ ]:
if all_after:
    target_id = all_after[0]["id"]

    print("Before update:")
    pp(m.get(target_id))

    m.update(target_id, "Alice is an advanced tennis player who trains daily.")
    print("\nAfter update:")
    pp(m.get(target_id))

    print("\nHistory after manual update:")
    pp(m.history(target_id))

In [ ]:
if len(all_after) > 1:
    del_id = all_after[-1]["id"]
    print(f"Deleting memory {del_id}")
    m.delete(del_id)
    print(f"Remaining memories: {len(m.get_all(user_id='alice'))}")

## 10. Multi-user isolation

Memories are isolated by `user_id` — Bob's memories won't appear in Alice's results.

In [ ]:
m.add(
    "I'm a software engineer working on distributed systems in Rust.",
    user_id="bob",
)

print("=== Bob's memories ===")
pp(m.get_all(user_id="bob"))

print("\n=== Alice's memories (unchanged) ===")
pp(m.get_all(user_id="alice"))

print(f"\n=== Search 'tennis' as Bob (expect nothing) ===")
pp(m.search("tennis", user_id="bob"))

## 11. Context-Aware Search

When `context` is enabled, mem7 auto-classifies each query's task type and adjusts scores based on memory type relevance. Preferences get demoted during troubleshooting tasks.

In [ ]:
print("=== Memory types ===")
for mem in m.get_all(user_id="alice"):
    mt = mem.get("memory_type", "unknown")
    print(f"  [{mt}] {mem['text']}")

print("\n=== Auto-classified search ===")
pp(m.search("What sports does Alice play?", user_id="alice", limit=3))

print("\n=== Override: task_type=troubleshooting (preferences demoted) ===")
pp(m.search("What sports does Alice play?", user_id="alice", limit=3, task_type="troubleshooting"))

## 12. Cleanup

In [ ]:
m.reset()
print("All data cleared ✓")